In [ ]:
!pip install mne -q
import os, re, glob, random, subprocess, shutil
import numpy as np
import mne
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
PARTS_DIR = os.path.join(SAVE_DIR, "chbmit_parts")
SAVE_NAME = "CHBMIT_crosspatient.npy"
DATA_ROOT = "physionet.org/files/chbmit/1.0.0"

FS = 256
WINDOW_SEC = 10
WINDOW = FS * WINDOW_SEC
GUARD_SEC = 120
NONSEIZURE_RATIO = 2.0
SEED = 0

TARGET = ["FP1-F7","F7-T7","T7-P7","P7-O1","FP1-F3","F3-C3","C3-P3","P3-O1",
          "FP2-F4","F4-C4","C4-P4","P4-O2","FP2-F8","F8-T8","T8-P8","P8-O2",
          "FZ-CZ","CZ-PZ"]

PATIENTS = [f"chb{i:02d}" for i in range(1, 25)]
TEST_PATIENTS = ["chb20","chb22","chb23","chb24"]

random.seed(SEED); np.random.seed(SEED)
os.makedirs(PARTS_DIR, exist_ok=True)

In [ ]:
!pip install awscli -q

NEG_CAP = 3000

def parse_summary(path):
    seiz, cur, pending = {}, None, None
    for line in open(path):
        line = line.strip()
        if line.startswith("File Name"):
            cur = line.split(":", 1)[1].strip(); seiz.setdefault(cur, [])
        elif "Seizure" in line and "Start Time" in line:
            pending = int(re.findall(r"\d+", line.split(":")[-1])[0])
        elif "Seizure" in line and "End Time" in line and pending is not None:
            e = int(re.findall(r"\d+", line.split(":")[-1])[0])
            seiz[cur].append((pending, e)); pending = None
    return seiz

def pick_channels(raw):
    clean = {}
    for i, n in enumerate(raw.ch_names):
        c = re.sub(r"-\d+$", "", n.upper().replace(" ", ""))
        if c not in clean:
            clean[c] = i
    if all(t in clean for t in TARGET):
        return [clean[t] for t in TARGET]
    return None

def windows_from_file(path, seizures):
    raw = mne.io.read_raw_edf(path, preload=True, verbose="ERROR")
    idx = pick_channels(raw)
    if idx is None:
        return [], []
    if int(raw.info["sfreq"]) != FS:
        raw.resample(FS, verbose="ERROR")
    data = raw.get_data()[idx].astype(np.float32)
    del raw
    n = data.shape[1] // WINDOW
    pos, neg = [], []
    for i in range(n):
        t0, t1 = i * WINDOW_SEC, (i + 1) * WINDOW_SEC
        seg = data[:, i * WINDOW:(i + 1) * WINDOW].copy()
        is_seiz = any(s < t1 and t0 < e for s, e in seizures)
        near = any(t0 < e + GUARD_SEC and t1 > s - GUARD_SEC for s, e in seizures)
        if is_seiz:
            pos.append(seg)
        elif not near:
            neg.append(seg)
    return pos, neg

def process_patient(p):
    pdir = os.path.join(DATA_ROOT, p)
    subprocess.run(["aws","s3","sync",
                    f"s3://physionet-open/chbmit/1.0.0/{p}/",
                    f"{pdir}/","--no-sign-request","--only-show-errors"],
                   timeout=1800)
    summ = os.path.join(pdir, f"{p}-summary.txt")
    if not os.path.exists(summ):
        return None
    seiz_map = parse_summary(summ)
    pos_all, neg_all, seen = [], [], 0
    for edf in sorted(glob.glob(os.path.join(pdir, "*.edf"))):
        pos, neg = windows_from_file(edf, seiz_map.get(os.path.basename(edf), []))
        pos_all += pos
        for w in neg:
            if len(neg_all) < NEG_CAP:
                neg_all.append(w)
            else:
                j = random.randint(0, seen)
                if j < NEG_CAP:
                    neg_all[j] = w
            seen += 1
    shutil.rmtree(pdir, ignore_errors=True)
    if len(pos_all) == 0:
        return None
    random.shuffle(neg_all)
    neg_all = neg_all[:int(NONSEIZURE_RATIO * len(pos_all))]
    X = np.asarray(pos_all + neg_all, np.float32)
    y = np.asarray([1]*len(pos_all) + [0]*len(neg_all), np.int64)
    np.save(os.path.join(PARTS_DIR, f"{p}.npy"), {"X": X, "y": y}, allow_pickle=True)
    return len(pos_all), len(neg_all)

for p in PATIENTS:
    if os.path.exists(os.path.join(PARTS_DIR, f"{p}.npy")):
        print(p, "already done, skipping", flush=True); continue
    try:
        r = process_patient(p)
        print(p, "seizure/nonseizure", r, flush=True)
    except subprocess.TimeoutExpired:
        print(p, "download timed out, skipping", flush=True)

In [ ]:
Xtr, ytr, Xte, yte = [], [], [], []
for p in PATIENTS:
    fp = os.path.join(PARTS_DIR, f"{p}.npy")
    if not os.path.exists(fp):
        continue
    d = np.load(fp, allow_pickle=True).item()
    if p in TEST_PATIENTS:
        Xte.append(d["X"]); yte.append(d["y"])
    else:
        Xtr.append(d["X"]); ytr.append(d["y"])

def stack(Xs, ys):
    if not Xs:
        return np.zeros((0, len(TARGET), WINDOW), np.float32), np.zeros((0,), np.int64)
    X = np.concatenate(Xs); y = np.concatenate(ys)
    order = np.random.permutation(len(X))
    return X[order], y[order]

data = {"train": dict(zip(["X","y"], stack(Xtr, ytr))),
        "test":  dict(zip(["X","y"], stack(Xte, yte)))}
np.save(os.path.join(SAVE_DIR, SAVE_NAME), data, allow_pickle=True)
print("saved", data["train"]["X"].shape, data["test"]["X"].shape)

In [ ]:
d = np.load(os.path.join(SAVE_DIR, SAVE_NAME), allow_pickle=True).item()
for s in ["train", "test"]:
    X, y = d[s]["X"], d[s]["y"]
    print(s, X.shape, "seizure fraction", round(float(y.mean()), 3))